In [14]:
! pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [15]:
import whisper
model = whisper.load_model("base")

In [16]:
model.device

device(type='cuda', index=0)

In [17]:

import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [18]:
import whisper
model = whisper.load_model("base")

In [19]:
from IPython.display import Audio

Audio("Conan.mp3") #30 secounds clip

In [21]:
import whisper

model = whisper.load_model("large-v3")

audio = whisper.load_audio("Conan.mp3")
audio = whisper.pad_or_trim(audio)

mel = whisper.log_mel_spectrogram(
    audio,
    n_mels=model.dims.n_mels
).to(model.device)

# Specifing Arabic directly
options = whisper.DecodingOptions(
    language="ar",
    task="transcribe"
)

result = whisper.decode(model, mel, options)

print(result.text)

ما أصعب أن يغيب شخص عزيز عنك فغيابه الطويل قد يجعله ينساك وما أصعب الانتظار أيضاً ولكنني سأبقى أنتظر تقي بسينشي فهو وفي ولن يتغير وأنا واثق تماماً بأنه لن ينساك صدقيني إن انتظارك هذا لن يطول أبداً عندما أنظر إليك يا كونان أتذكر سينشي


In [22]:
! pip install gradio -q

In [23]:
def transcribe(audio):
    audio = whisper.load_audio(audio)
    audio = whisper.pad_or_trim(audio)

    mel = whisper.log_mel_spectrogram(audio).to(model.device)

    _, probs = model.detect_language(mel)
    print(f"Detected language: {max(probs, key=probs.get)}")

    options = whisper.DecodingOptions()
    result = whisper.decode(model, mel, options)

    return result.text

In [24]:
import gradio as gr

gr.Interface(
    fn=transcribe,
    inputs=[
        gr.Audio(
            type="filepath"
        ),
    ],
    outputs=[
        gr.Textbox(),
    ],
    live=True
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://43c4837c1f3e61ad9f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [25]:
!pip install transformers -q


In [27]:
!pip install -U transformers accelerate datasets[audio]

## pipeline and tourch


In [28]:
from transformers import pipeline
import torch

p = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=0 if torch.cuda.is_available() else -1
)

result = p(
    "Conan.mp3",
    generate_kwargs={
        "language": "arabic",
        "task": "transcribe"
    }
)

print(result["text"])

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 ما أصعب أن يغيب شخص عزيز عنك فغيابه الطويل قد يجعله ينساك وما أصعب الانتظار أيضا ولكنني سأبقى أنتظر تقيب سينشي فهو وفين ولا يتغير وانا وافقا تماما بأنه لا ينساكي صدقيني إن انتظارك هذا لا يطول أبدا عندما أنظر إلىك يا كونا أتذكر سينشي


## More Accurate model Hugging face

In [1]:
!pip install -U "transformers>=5.4.0" accelerate soundfile librosa sentencepiece protobuf

In [2]:
from huggingface_hub import login

login()

In [3]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '6ab06468c65439b75592035c', 'name': 'lol120', 'fullname': 'Lujain Yousef Mohmmad Heresh', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1793491200, 'isPro': False, 'avatarUrl': '/avatars/56e58ae9e1baf85f0e87ceff9a56edbc.svg', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-11-04T18:54:21.000Z'}}


In [4]:
from huggingface_hub import login
login()

In [5]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="CohereLabs/cohere-transcribe-arabic-07-2026",
    filename="config.json"
)

print("Access OK")

config.json:   0%|          | 0.00/3.42k [00:00<?, ?B/s]

Access OK


In [6]:
import torch
from transformers import AutoProcessor, CohereAsrForConditionalGeneration
from transformers.audio_utils import load_audio

model_id = "CohereLabs/cohere-transcribe-arabic-07-2026"

# Load processor and model
processor = AutoProcessor.from_pretrained(model_id)

model = CohereAsrForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto"
)

# Load audio
audio_path = "Conan.mp3"

audio = load_audio(
    audio_path,
    sampling_rate=16000
)

# Prepare audio
inputs = processor(
    audio=audio,
    sampling_rate=16000,
    return_tensors="pt",
    language="ar"
)

# Save chunk information for long audio
audio_chunk_index = inputs.get("audio_chunk_index")

# Move inputs to GPU
inputs.to(model.device, dtype=model.dtype)

# Generate transcription
outputs = model.generate(
    **inputs,
    max_new_tokens=256
)

# Decode
text = processor.decode(
    outputs,
    skip_special_tokens=True,
    audio_chunk_index=audio_chunk_index,
    language="ar"
)[0]

print(text)

processor_config.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.13GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2150 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

 ما أصعب أن يغيب شخص عزيز عنك. فغيابه الطويل قد يجعله ينساك. وما أصعب الانتظار أيضاً. ولكنني سأبقى أنتظر. ثقي بسينشي فهو وفي ولن يتغير. وأنا واثق تماماً بأنه لن ينساك. صدقيني إن انتظارك هذا لن يطول أبداً. عندما أنظر إليك يا كونان أتذكر سينشي.


# background noise

In [19]:
from IPython.display import Audio

Audio("Friendss.mp3")

In [16]:
import torch
from transformers import AutoProcessor, CohereAsrForConditionalGeneration
from transformers.audio_utils import load_audio

model_id = "CohereLabs/cohere-transcribe-arabic-07-2026"

# Load processor and model
processor = AutoProcessor.from_pretrained(model_id)

model = CohereAsrForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto"
)

# Load audio
audio_path = "Friendss.mp3"

audio = load_audio(
    audio_path,
    sampling_rate=16000
)

# Prepare audio
inputs = processor(
    audio=audio,
    sampling_rate=16000,
    return_tensors="pt",
    language="ar"
)

# Save chunk information for long audio
audio_chunk_index = inputs.get("audio_chunk_index")

# Move inputs to GPU
inputs.to(model.device, dtype=model.dtype)

# Generate transcription
outputs = model.generate(
    **inputs,
    max_new_tokens=256
)

# Decode
text = processor.decode(
    outputs,
    skip_special_tokens=True,
    audio_chunk_index=audio_chunk_index,
    language="ar"
)[0]

print(text)

Loading weights:   0%|          | 0/2150 [00:00<?, ?it/s]

 مرت الأيام بسرعة، أما زلت تذكر اليوم الأول للقائنا؟ قضينا معا فترة قصيرة، كنت خلالها أفضل شخص تعرفت إليه في حياتي كلها. أعدك أن تكون صداقتنا حقيقية إلى الأبد.


# Long text

In [21]:
from IPython.display import Audio

Audio("Hunter.mp3")

In [17]:
import torch
from transformers import AutoProcessor, CohereAsrForConditionalGeneration
from transformers.audio_utils import load_audio

model_id = "CohereLabs/cohere-transcribe-arabic-07-2026"

# Load processor and model
processor = AutoProcessor.from_pretrained(model_id)

model = CohereAsrForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto"
)

# Load audio
audio_path = "Hunter.mp3"

audio = load_audio(
    audio_path,
    sampling_rate=16000
)

# Prepare audio
inputs = processor(
    audio=audio,
    sampling_rate=16000,
    return_tensors="pt",
    language="ar"
)

# Save chunk information for long audio
audio_chunk_index = inputs.get("audio_chunk_index")

# Move inputs to GPU
inputs.to(model.device, dtype=model.dtype)

# Generate transcription
outputs = model.generate(
    **inputs,
    max_new_tokens=256
)

# Decode
text = processor.decode(
    outputs,
    skip_special_tokens=True,
    audio_chunk_index=audio_chunk_index,
    language="ar"
)[0]

print(text)

Loading weights:   0%|          | 0/2150 [00:00<?, ?it/s]

 لست مضطراً إلى قتال هذا السفاح يا كيلوا. لنستسلم ونعلن انسحابنا. انتظر يا كيلوا. كيلوا. كيف تريد النزال؟ نزال؟ هل تمزح معي؟ ما سيحدث الآن هو جريمة نكرهه وستكون أنت الضحية. سأمزقه الآن بيدي. قلبي أشعر بالبرد
